# Titanic Survival Prediction

This notebook builds a machine learning workflow to predict passenger survival.

The first part focuses on creating useful features from the raw passenger data. Later sections will cover preprocessing, model training, evaluation, and model interpretation.


## 1. Setup and Data Loading

First, I import the libraries I need and load the training and test datasets.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import warnings
warnings.filterwarnings("ignore")


In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)


Training data shape: (891, 12)
Test data shape: (418, 11)


## 2. Initial Data Check

Before creating features, I checked the main columns and the data types.


In [3]:
train.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
train.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## 3. Feature Engineering

I will create a few features that may give the models more useful information than the raw columns alone.

The features are:

- `Title`
- `FamilySize`
- `IsAlone`
- `AgeBin`
- `FareBin`
- `Deck`
- `CabinKnown`


### Extract Title

The title is taken from the passenger's name.

Titles such as `Mr`, `Mrs`, `Miss`, and `Master` can give some information about a passenger's age and social group.


In [5]:
train["Title"] = train["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
test["Title"] = test["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()

train["Title"].value_counts()


Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Mlle              2
Major             2
Col               2
the Countess      1
Capt              1
Ms                1
Sir               1
Lady              1
Mme               1
Don               1
Jonkheer          1
Name: count, dtype: int64

### Create FamilySize

`FamilySize` shows how many people were travelling together in the passenger's immediate family.

The passenger is included in the total.


In [6]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

train["FamilySize"].describe()


count    891.000000
mean       1.904602
std        1.613459
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       11.000000
Name: FamilySize, dtype: float64

### Create IsAlone

`IsAlone` shows whether a passenger was travelling alone.

A value of `1` means the passenger was alone, while `0` means they were travelling with family.


In [7]:
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

train["IsAlone"].value_counts()


IsAlone
1    537
0    354
Name: count, dtype: int64

### Create AgeBin

Age is grouped into a few simple categories.

I am leaving missing ages as missing for now. They will be handled later by the preprocessing pipeline.


In [8]:
age_bins = [0, 12, 18, 35, 60, np.inf]
age_labels = ["Child", "Teen", "Young Adult", "Adult", "Senior"]

train["AgeBin"] = pd.cut(
    train["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

test["AgeBin"] = pd.cut(
    test["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

train["AgeBin"].value_counts(dropna=False)


AgeBin
Young Adult    358
Adult          195
NaN            177
Teen            70
Child           69
Senior          22
Name: count, dtype: int64

### Create FareBin

Fare is grouped into four categories using the quartiles of the training data.

The bin edges come from the training data so the test data does not influence them.


In [9]:
fare_bins = train["Fare"].quantile([0, 0.25, 0.50, 0.75, 1.00]).values
fare_bins = np.unique(fare_bins)

fare_labels = ["Low", "Medium", "High", "Very High"][:len(fare_bins) - 1]

train["FareBin"] = pd.cut(
    train["Fare"],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

test["FareBin"] = pd.cut(
    test["Fare"],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

train["FareBin"].value_counts(dropna=False)


FareBin
Medium       224
Low          223
High         222
Very High    222
Name: count, dtype: int64

### Extract Deck

The first character of `Cabin` is used as the deck.

Missing cabin information is labelled as `Unknown` instead of creating a made-up deck.


In [10]:
train["Deck"] = train["Cabin"].str[0].fillna("Unknown")
test["Deck"] = test["Cabin"].str[0].fillna("Unknown")

train["Deck"].value_counts()


Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

### Create CabinKnown

`CabinKnown` shows whether cabin information is available.

This keeps the information about missing cabin records without trying to guess the missing cabin.


In [11]:
train["CabinKnown"] = train["Cabin"].notna().astype(int)
test["CabinKnown"] = test["Cabin"].notna().astype(int)

train["CabinKnown"].value_counts()


CabinKnown
0    687
1    204
Name: count, dtype: int64

### Check the New Features

I will check the new columns before moving on to the preprocessing stage.


In [12]:
new_features = [
    "Title",
    "FamilySize",
    "IsAlone",
    "AgeBin",
    "FareBin",
    "Deck",
    "CabinKnown"
]

train[new_features].head()


,Title,FamilySize,IsAlone,AgeBin,FareBin,Deck,CabinKnown
0,Mr,2,0,Young Adult,Low,Unknown,0
1,Mrs,2,0,Adult,Very High,C,1
2,Miss,1,1,Young Adult,Medium,Unknown,0
3,Mrs,2,0,Young Adult,Very High,C,1
4,Mr,1,1,Young Adult,Medium,Unknown,0


In [13]:
train[new_features].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Title       891 non-null    object  
 1   FamilySize  891 non-null    int64   
 2   IsAlone     891 non-null    int32   
 3   AgeBin      714 non-null    category
 4   FareBin     891 non-null    category
 5   Deck        891 non-null    object  
 6   CabinKnown  891 non-null    int32   
dtypes: category(2), int32(2), int64(1), object(2)
memory usage: 30.1+ KB


In [14]:
train[new_features].isnull().sum()


Title           0
FamilySize      0
IsAlone         0
AgeBin        177
FareBin         0
Deck            0
CabinKnown      0
dtype: int64

## 4. Next Step

The feature engineering is now complete.

The next step is to split the training data and build a leakage-free preprocessing pipeline using `ColumnTransformer` and `Pipeline`.


## 4. Prepare the Data

The `Survived` column is the target.

I will keep the original training data for the machine learning workflow and separate the target before splitting the data.


In [15]:
X = train.drop(columns=["Survived"])
y = train["Survived"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


Features shape: (891, 18)
Target shape: (891,)


## 5. Train-Test Split

I will split the training data into training and validation sets.

Stratification is used so the proportion of survivors stays similar in both sets.


In [16]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Validation set:", X_valid.shape)


Training set: (712, 18)
Validation set: (179, 18)


## 6. Leakage-Free Preprocessing

The raw data contains both numerical and categorical columns.

Instead of filling missing values or creating dummy variables before the split, I will put these steps inside a `ColumnTransformer`.

This means the preprocessing is learned from the training data only.

The same preprocessing can then be used on validation or test data without fitting it again.


### Select Columns

The model will use the original numerical variables together with the new features created earlier.

`PassengerId` and `Name` are not used directly because they are identifiers or raw text fields that have already been used to create more useful features.


In [17]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone",
    "CabinKnown"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title",
    "AgeBin",
    "FareBin",
    "Deck"
]

X_train[numeric_features + categorical_features].head()


,Pclass,Age,SibSp,Parch,Fare,FamilySize,IsAlone,CabinKnown,Sex,Embarked,Title,AgeBin,FareBin,Deck
692,3,NaN,0,0,56.4958,1,1,0,male,S,Mr,NaN,Very High,Unknown
481,2,NaN,0,0,0.0000,1,1,0,male,S,Mr,NaN,Low,Unknown
527,1,NaN,0,0,221.7792,1,1,1,male,S,Mr,NaN,Very High,C
855,3,18.0,0,1,9.3500,2,0,0,female,S,Mrs,Teen,Medium,Unknown
801,2,31.0,1,1,26.2500,3,0,0,female,S,Mrs,Young Adult,High,Unknown


### Numerical Preprocessing

For numerical columns, missing values are filled with the median.

The median is calculated by the pipeline from the training data. The values are then scaled using `StandardScaler`.


In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


### Categorical Preprocessing

For categorical columns, missing values are filled with the most common value.

The categories are then converted into numeric columns using one-hot encoding.

Unknown categories in validation or test data are ignored instead of causing an error.


In [19]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


### Combine the Preprocessing Steps

`ColumnTransformer` applies the correct preprocessing to each type of column.


In [20]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


## 7. Build the Model Pipeline

The preprocessing and model are placed in one `Pipeline`.

This is important because the preprocessing is fitted as part of the model workflow rather than separately on the full dataset.


In [21]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])


## 8. Fit the Pipeline

The pipeline is fitted using the training data only.

The imputer, scaler, and encoder learn their values from `X_train`.


In [22]:
logistic_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare',
                                                   'FamilySize', 'IsAlone',
                                                   'CabinKnown']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked', 'Title',
                                                   'AgeBin', 'FareBin',
                                                   'Deck'])])),
                ('model', LogisticRegression(max_iter=1000))])

## 9. Check the Validation Set

The fitted pipeline can now make predictions on the validation data.

No separate imputation or encoding is needed.


In [23]:
y_pred = logistic_model.predict(X_valid)
y_prob = logistic_model.predict_proba(X_valid)[:, 1]

print("Accuracy:", round(accuracy_score(y_valid, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_valid, y_prob), 4))
print("F1 Score:", round(f1_score(y_valid, y_pred), 4))


Accuracy: 0.838
ROC-AUC: 0.8665
F1 Score: 0.7852


## 10. Preprocessing Check

The preprocessing and model are now working together in one pipeline.

The important point is that the validation data was transformed using the preprocessing fitted on the training data. It was not used to calculate the median, scaling values, or category mapping.

This gives us a leakage-free starting point for the model comparison and cross-validation steps.


In [24]:
print(logistic_model)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare',
                                                   'FamilySize', 'IsAlone',
                                                   'CabinKnown']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                     

## 11. Check the Pipeline with Cross-Validation

The preprocessing is inside the model pipeline, so it is fitted separately inside each training fold.

This is the main reason for using `Pipeline` here instead of preprocessing the full dataset first.


In [25]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []

for train_idx, valid_idx in cv.split(X, y):
    X_fold_train = X.iloc[train_idx]
    X_fold_valid = X.iloc[valid_idx]
    y_fold_train = y.iloc[train_idx]
    y_fold_valid = y.iloc[valid_idx]

    fold_model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ])

    fold_model.fit(X_fold_train, y_fold_train)
    fold_pred = fold_model.predict(X_fold_valid)

    cv_scores.append(accuracy_score(y_fold_valid, fold_pred))

print("Cross-validation accuracy:", [round(score, 4) for score in cv_scores])
print("Mean accuracy:", round(np.mean(cv_scores), 4))


Cross-validation accuracy: [0.8436, 0.8371, 0.8034, 0.8315, 0.8315]
Mean accuracy: 0.8294


## 12. Next Step

The leakage-free preprocessing is now in place and has been checked with stratified cross-validation.

The next step is to train and compare several models using the same preprocessing setup.
